# Binary classification

The simplest use of sLTN: learn one predicate `A` from positive and negative examples. The knowledge base says "every positive example is `A`" and "no negative example is `A`"; training a network to satisfy it is ordinary supervised learning written as logic.

## Problem

100 points are drawn uniformly from the unit square. A point is positive when it lies inside the disc of radius 0.3 centred at (0.5, 0.5):

$$\|x - (0.5, 0.5)\|^2 < 0.3^2.$$

The task is to learn a predicate $A$ such that

$$\mathcal{K} = \{\ \forall x_{pos}: A(x_{pos}),\quad \forall x_{neg}: \lnot A(x_{neg})\ \}.$$

In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt
import sltn
from sltn import Signature, Interpretation, KB

torch.manual_seed(0)
np.random.seed(0)
device = "cpu"

## Data

75 points for training, 25 for testing.

In [ ]:
nr_samples = 100
dataset = torch.rand((nr_samples, 2))
labels = torch.sum((dataset - torch.tensor([0.5, 0.5])) ** 2, dim=1) < 0.09

nr_train = 75
x_train, y_train = dataset[:nr_train].to(device), labels[:nr_train].to(device)
x_test, y_test = dataset[nr_train:].to(device), labels[nr_train:].to(device)

plt.figure(figsize=(4, 4))
plt.scatter(*dataset[labels].T, label="A")
plt.scatter(*dataset[~labels].T, label="not A")
plt.title("Ground truth")
plt.legend()
plt.show()

## 1 — Signature

The signature declares the vocabulary: a sort `Point`, the predicate `A` over points, and three variables of sort `Point`. `x_pos` and `x_neg` will hold the positive and negative examples of a batch, `x` any set of points to be queried.

In [ ]:
sig = Signature("binary_clf")
sig.sort("Point")
sig.variable("x_pos", "Point")
sig.variable("x_neg", "Point")
sig.variable("x", "Point")
sig.predicate("A", ["Point"])
print(sig)

## 2 — Knowledge base

Two clauses, one per kind of example:

- `forall x_pos: A(x_pos)`: every positive example is `A`;
- `forall x_neg: not A(x_neg)`: no negative example is `A`.

Splitting the examples into two variables is how LTN encodes labels: each variable is grounded with the examples of one class. `∀` is pMeanError (p = 2), a soft minimum, and `KB` combines the clauses with `SatAgg`.

In [ ]:
kb = KB([
    sig.parse("forall x_pos: A(x_pos)"),
    sig.parse("forall x_neg: not A(x_neg)"),
])
print(kb)

## 3 — Model

`A` is a small MLP ending in a sigmoid, so its output is a truth degree in [0, 1].

In [ ]:
class ModelA(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.net = torch.nn.Sequential(
            torch.nn.Linear(2, 16), torch.nn.ELU(),
            torch.nn.Linear(16, 16), torch.nn.ELU(),
            torch.nn.Linear(16, 1), torch.nn.Sigmoid(),
        )

    def forward(self, x):
        return self.net(x)

## 4 — Interpretation

The interpretation grounds the symbols: `Point` is a vector of 2 reals, `A` is the network. Since `ModelA` is an `nn.Module`, it is registered in the interpretation: `interp.parameters()` returns its weights, and `interp.train()` / `interp.eval()` switch its mode. The variables are grounded later, batch by batch.

In [ ]:
interp = Interpretation(sig).to(device)
interp["Point"] = sltn.Type("Point", shape=(2,))
interp["A"] = ModelA()

## 5 — Metrics

- **Satisfaction**: the truth value of the KB on a set of examples.
- **Accuracy**: `A(x)` thresholded at 0.5, compared with the labels.

Both are computed in eval mode and without gradients.

In [ ]:
def satisfaction(x, y):
    interp["x_pos"], interp["x_neg"] = x[y], x[~y]
    return interp(kb).value.item()


def accuracy(x, y):
    interp["x"] = x
    predicted = interp("A(x)").value.squeeze(-1) > 0.5
    return (predicted == y).float().mean().item()


def evaluate():
    interp.eval()
    with torch.no_grad():
        return (satisfaction(x_train, y_train), satisfaction(x_test, y_test),
                accuracy(x_train, y_train), accuracy(x_test, y_test))

## 6 — Training

The loss is $1 - \mathrm{SatAgg}(\mathcal{K})$: maximising the satisfaction of the KB. Each step grounds `x_pos` and `x_neg` with the positive and negative examples of a minibatch of 64 points.

In [ ]:
optimizer = torch.optim.Adam(interp.parameters(), lr=0.001)

for epoch in range(1000):
    interp.train()
    for idx in torch.randperm(nr_train).split(64):
        x, y = x_train[idx], y_train[idx]
        interp["x_pos"], interp["x_neg"] = x[y], x[~y]
        optimizer.zero_grad()
        loss = 1 - interp(kb).value
        loss.backward()
        optimizer.step()

    if epoch % 200 == 0 or epoch == 999:
        sat_tr, sat_te, acc_tr, acc_te = evaluate()
        print(f"epoch {epoch:4d} | sat train {sat_tr:.3f} test {sat_te:.3f} "
              f"| acc train {acc_tr:.3f} test {acc_te:.3f}")

## 7 — Results

The plot shows `A(x)` on a grid over the unit square, with the training (circles) and test (crosses) points coloured by their true class. The learned region matches the disc, and the test accuracy is close to that on the training set.

In [ ]:
grid = torch.stack(torch.meshgrid(torch.linspace(0, 1, 100), torch.linspace(0, 1, 100), indexing="xy"), dim=-1)

interp.eval()
with torch.no_grad():
    interp["x"] = grid.reshape(-1, 2).to(device)
    a_grid = interp("A(x)").value.reshape(100, 100).cpu()

plt.figure(figsize=(5, 4.5))
plt.contourf(grid[..., 0], grid[..., 1], a_grid, levels=20, cmap="viridis", vmin=0, vmax=1)
plt.colorbar(label="A(x)")
plt.contour(grid[..., 0], grid[..., 1], a_grid, levels=[0.5], colors="white")
plt.scatter(*dataset[:nr_train].T, c=labels[:nr_train], cmap="coolwarm", edgecolors="k", s=25)
plt.scatter(*dataset[nr_train:].T, c=labels[nr_train:], cmap="coolwarm", marker="x", s=40)
plt.gca().add_patch(plt.Circle((0.5, 0.5), 0.3, fill=False, ls="--", color="red"))
plt.title("A(x); white: A = 0.5, dashed red: true disc")
plt.show()

## Summary

- Labels are encoded by grounding one variable per class (`x_pos`, `x_neg`) and writing one clause per class.
- The loss is $1 - $ the satisfaction of the KB; with only these two clauses it is a standard classification loss.
- A network grounding a predicate is an `nn.Module` registered in the interpretation: train it with `interp.parameters()`, and evaluate with `interp.eval()` and `torch.no_grad()`.
- The same KB can be queried on any data by grounding a variable (`x`) and evaluating a formula (`A(x)`).